# 01. 데이터 수집 및 전처리

NOAA MarineCadastre AIS 데이터(2022-01-01)를 로드하고 전처리한다.

- 원본: ~744MB, 약 1,000만 행
- 전처리 후 경량 parquet 파일 생성 → 이후 노트북에서 활용

In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.append('..')

print('pandas:', pd.__version__)

## 1.1 데이터 로드

필요한 컬럼만 선택하고 dtype 지정하여 메모리를 절약한다.

In [ ]:
USE_COLS = ['MMSI', 'BaseDateTime', 'LAT', 'LON', 'SOG', 'COG', 'Heading',
            'VesselType', 'VesselName', 'Length', 'Width', 'Status']

df = pd.read_csv(
    '../data/raw/AIS_2022_01_01.csv',
    usecols=USE_COLS,
    dtype={'MMSI': 'int64', 'VesselType': 'float32', 'SOG': 'float32',
           'COG': 'float32', 'Heading': 'float32'},
    parse_dates=['BaseDateTime']
)

print(f'로드 완료: {len(df):,} rows, {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
df.head()

## 1.2 기본 정보 확인

In [ ]:
print('=== 데이터 타입 ===')
print(df.dtypes)
print(f'\n=== 결측치 ===')
print(df.isnull().sum())
print(f'\n고유 선박 수: {df["MMSI"].nunique():,}')
print(f'시간 범위: {df["BaseDateTime"].min()} ~ {df["BaseDateTime"].max()}')

## 1.3 전처리

In [ ]:
print(f'전처리 전: {len(df):,}')

# 1) 유효하지 않은 좌표 제거
df = df[(df['LAT'].between(-90, 90)) & (df['LON'].between(-180, 180))]
df = df[(df['LAT'] != 0) & (df['LON'] != 0)]
print(f'좌표 필터링 후: {len(df):,}')

# 2) 비정상 속도 제거 (음수, 102.3 이상 = AIS 미수신)
df = df[(df['SOG'] >= 0) & (df['SOG'] < 102.3)]
print(f'속도 필터링 후: {len(df):,}')

# 3) 타임스탬프 결측 제거
df = df.dropna(subset=['BaseDateTime'])
print(f'시간 필터링 후: {len(df):,}')

# 4) 선박별 시간순 정렬
df = df.sort_values(['MMSI', 'BaseDateTime']).reset_index(drop=True)
print(f'\n전처리 완료: {len(df):,}')

## 1.4 샘플링 및 저장

활동이 많은 선박 상위 500척을 추출하여 경량 parquet 파일을 생성한다.

In [ ]:
# 레코드 수 상위 500척 추출
top_vessels = df['MMSI'].value_counts().head(500).index
df_sample = df[df['MMSI'].isin(top_vessels)].copy()

print(f'상위 500척 추출: {len(df_sample):,} rows')
print(f'고유 선박: {df_sample["MMSI"].nunique()}')

# parquet 저장
df_sample.to_parquet('../data/ais_cleaned.parquet', index=False)

import os
fsize = os.path.getsize('../data/ais_cleaned.parquet') / 1e6
print(f'저장 완료: data/ais_cleaned.parquet ({fsize:.1f} MB)')

In [ ]:
# 검증
df_check = pd.read_parquet('../data/ais_cleaned.parquet')
print(f'Parquet 로드 확인: {len(df_check):,} rows, {df_check["MMSI"].nunique()} vessels')
df_check.head()